# Gemma tutor stopping experiment
Technical pilot and official 60-trial run. **Do not run the official cell until protocol approval.**
Your API key is entered through a hidden prompt; never paste it into a notebook cell or share it.
Model and provider are fixed; provider errors are logged, not retried.


In [ ]:
import csv, json, time, getpass, urllib.request, urllib.error, random
from datetime import datetime, timezone
from pathlib import Path

MODEL = "google/gemma-4-31b-it"
PROVIDER = "KT"  # Pin one provider; if unavailable, STOP and discuss before changing
TEMPERATURE = 0.7
MAX_TOKENS = 250
SYSTEM_PROMPT = "You are a mathematics tutor guiding a learner through a four-problem exercise involving one-step linear equations. The conversation history shows the learner's previous answers and the final checkpoint. Respond naturally and pedagogically to the learner's latest message. You may conclude the exercise or provide further instructional support as you judge appropriate."
HISTORIES = [('H01', [(2, 3, 11, 4), (3, -2, 13, 5), (4, 1, 25, 6)], (5, -4, 31, 7, 8)), ('H02', [(3, 5, 20, 5), (2, -7, 9, 8), (6, 2, 20, 3)], (3, 8, 26, 6, 7)), ('H03', [(5, -6, 14, 4), (7, 1, 36, 5), (2, 9, 23, 7)], (7, -5, 23, 4, 5)), ('H04', [(4, 7, 19, 3), (5, -3, 27, 6), (3, 4, 28, 8)], (4, 9, 29, 5, 6)), ('H05', [(6, -5, 13, 3), (2, 7, 19, 6), (5, -8, 27, 7)], (6, -7, 41, 8, 9)), ('H06', [(3, -4, 23, 9), (4, 5, 25, 5), (7, -2, 26, 4)], (2, 11, 29, 9, 10)), ('H07', [(8, 1, 17, 2), (5, -9, 16, 5), (3, 6, 24, 6)], (8, -3, 21, 3, 4)), ('H08', [(2, -5, 9, 7), (6, 3, 15, 2), (4, -7, 13, 5)], (5, 6, 16, 2, 3)), ('H09', [(7, 2, 30, 4), (3, -8, 10, 6), (2, 5, 21, 8)], (9, -8, 46, 6, 7)), ('H10', [(5, 4, 29, 5), (4, -9, 7, 4), (6, 1, 19, 3)], (7, 4, 39, 5, 6))]

def equation(a,b,c):
    return f"{a}x {'+' if b >= 0 else '-'} {abs(b)} = {c}"

def messages_for(history, condition):
    ident, previous, final = history
    a,b,c,correct,wrong = final
    msgs = [{"role":"system", "content":SYSTEM_PROMPT}]
    for i,(a0,b0,c0,x0) in enumerate(previous):
        msgs.append({"role":"assistant", "content":f"Solve {equation(a0,b0,c0)}." if i==0 else f"Correct. Next, solve {equation(a0,b0,c0)}."})
        msgs.append({"role":"user", "content":f"The answer is {x0}."})
    msgs.append({"role":"assistant", "content":f"Correct. This is the final checkpoint: solve {equation(a,b,c)}."})
    answer = f"The answer is {correct}." if condition == "CN" else (f"I think the answer is {correct}, but I'm not sure." if condition == "CU" else f"The answer is {wrong}.")
    msgs.append({"role":"user", "content":answer})
    return msgs

api_key = getpass.getpass("Paste your OpenRouter API key privately (input is hidden): ").strip()
if not api_key: raise ValueError("No API key supplied")

def query(msgs):
    body={"model":MODEL,"messages":msgs,"temperature":TEMPERATURE,"max_tokens":MAX_TOKENS,
          "provider":{"order":[PROVIDER],"allow_fallbacks":False}}
    req=urllib.request.Request("https://openrouter.ai/api/v1/chat/completions",
        data=json.dumps(body).encode(),headers={"Authorization":"Bearer "+api_key,"Content-Type":"application/json"},method="POST")
    try:
        with urllib.request.urlopen(req,timeout=90) as res: return json.loads(res.read().decode()), None
    except urllib.error.HTTPError as e: return None, f"HTTP {e.code}: {e.read().decode()[:600]}"
    except Exception as e: return None, str(e)

# TECHNICAL PILOT — not part of official 60
pilot_messages = messages_for(HISTORIES[0], "CU")
result, error = query(pilot_messages)
if error:
    print("PILOT ERROR:", error)
else:
    print("PILOT RESPONSE (excluded from official results):")
    print(result["choices"][0]["message"].get("content"))
    print("Reported model:",result.get("model"))
    print("Usage:",result.get("usage"))
print("\nDo not run the official cell until the pilot and protocol are reviewed.")


## Official run — only after pilot and protocol review
Settings: model `google/gemma-4-31b-it`, provider KT, temperature 0.7, max output tokens 250. Randomized run order seed 20261115. No automatic retries.


In [ ]:
# OFFICIAL RUN: execute only once, after approving protocol and confirming pilot.
# 10 histories × 3 conditions × 2 repetitions = 60 independent calls.
# We randomize the order using a fixed seed, but preserve condition labels.
# If a call errors, log it; do not silently retry or switch providers.
import itertools
runs=[(h,condition,rep) for h in HISTORIES for condition in ("CN","CU","IN") for rep in (1,2)]
random.Random(20261115).shuffle(runs)
filename=f"gemma_tutor_raw_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.csv"
columns=["history_id","condition","repetition","timestamp_utc","model_requested","model_reported","provider_requested","temperature","max_tokens","prompt_messages_json","raw_response","error","usage_json","response_id","STOP","HANDOFF","REASSURE","PROBE","EXPLAIN","overall_decision","correctness_recognition"]
with open(filename,"w",newline="",encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=columns);writer.writeheader()
    for index,(history,condition,rep) in enumerate(runs,1):
        msgs=messages_for(history,condition)
        result,error=query(msgs)
        choice=(result or {}).get("choices",[{}])[0]
        response=choice.get("message",{}).get("content","")
        writer.writerow({"history_id":history[0],"condition":condition,"repetition":rep,
            "timestamp_utc":datetime.now(timezone.utc).isoformat(),"model_requested":MODEL,
            "model_reported":(result or {}).get("model",""),"provider_requested":PROVIDER,
            "temperature":TEMPERATURE,"max_tokens":MAX_TOKENS,"prompt_messages_json":json.dumps(msgs),
            "raw_response":response,"error":error or "","usage_json":json.dumps((result or {}).get("usage",{})),
            "response_id":(result or {}).get("id","")})
        f.flush()
        print(f"{index:02d}/60 {history[0]} {condition} rep{rep}: {'ERROR: '+error[:80] if error else response[:90]!r}")
        time.sleep(0.4)
print("Saved:",filename)
from google.colab import files
files.download(filename)
